### **Assignment 5** — Compare Linear Regression With Regularization
Objective Understand when Linear Regression fails and regularization helps.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files  #Comment out if you ar runnin locally
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Again using The California Housing Dataset

In [ ]:
uploaded = files.upload()  #Comment out if you ar runnin locally

Saving Housing.csv to Housing.csv


In [3]:
df = pd.read_csv('Housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


Preprocessing

In [4]:
# Handle missing values
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].mean())

# One-hot encoding
df = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)

In [5]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Feature Scaling

In [7]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Linear Regression

In [8]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

Ridge Regression

In [9]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

Lasso Regression

In [10]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
lasso_pred = lasso.predict(X_test)

Eval fun

In [11]:
def evaluate(y_test, pred):
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    return mae, rmse, r2

Compare Models

In [12]:
lr_mae, lr_rmse, lr_r2 = evaluate(y_test, lr_pred)
ridge_mae, ridge_rmse, ridge_r2 = evaluate(y_test, ridge_pred)
lasso_mae, lasso_rmse, lasso_r2 = evaluate(y_test, lasso_pred)

comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso'],
    'MAE': [lr_mae, ridge_mae, lasso_mae],
    'RMSE': [lr_rmse, ridge_rmse, lasso_rmse],
    'R2': [lr_r2, ridge_r2, lasso_r2]
})

comparison

,Model,MAE,RMSE,R2
0,Linear Regression,50701.779031,70031.419920,0.625735
1,Ridge,50699.229197,70028.478685,0.625767
2,Lasso,50701.720648,70031.356334,0.625736


Checking Lasso Feature Selection

In [13]:
lasso_coef = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lasso.coef_
})

# Features removed (coefficient = 0)
lasso_removed = lasso_coef[lasso_coef['Coefficient'] == 0]

print("Features removed by Lasso:")
print(lasso_removed)

Features removed by Lasso:
Empty DataFrame
Columns: [Feature, Coefficient]
Index: []
